# ERK + OIS Simulation

10 cellules fondatrices dont **une seule porte une mutation BRAF GOF** dès le départ.  
Plugins : `ErkPathwayPlugin`, `MelanocyteOISPlugin`, `DummyDeathPlugin`, `DummyBirthPlugin`, `ConstantMutationRatePlugin`.  

**Définition des clones** : chaque événement d'acquisition d'un driver crée un clone distinct, identifié par la cellule fondatrice du clone — deux cellules portant BRAF GOF issues de lignées indépendantes forment deux clones différents.

In [1]:
from evoseer.core.config import PluginConfig, RateFunctionConfig, SimulationConfig
from evoseer.engine.gillespie import GillespieEngine
from evoseer.plugins.mutation_rate import ConstantMutationRatePlugin
from evoseer.plugins.pathway.erk import ErkPathwayPlugin
from evoseer.plugins.ois.melanocyte import MelanocyteOISPlugin
from evoseer.plugins.dummies import DummyDeathPlugin, DummyBirthPlugin
from evoseer.rate_functions.weighted_sum import WeightedSumRate
from evoseer.recording.recorder import Recorder
from evoseer.recording.exporter import SimulationExporter
from evoseer.services.mutation_generator import UniformMutationGenerator
from evoseer.services.mutation_store import InMemoryMutationStore, MutationRecord

## 1. Mutation store

- `BRAF_GOF_ID` : driver BRAF GOF, pathways ERK + OIS
- Passengers avec `d=0.005` (mort légère)
- `DENSITY_MUT_ID` : régulation densité-dépendante donnée à tous les fondateurs

In [2]:
N_PASS         = 5_000
BRAF_GOF_ID    = N_PASS + 1
NRAS_GOF_ID    = N_PASS + 2
DENSITY_MUT_ID = N_PASS + 100

def build_store() -> InMemoryMutationStore:
    store = InMemoryMutationStore()
    for i in range(N_PASS):
        store.add_mutation(MutationRecord(
            mutation_id=i, is_driver=False, extra={"d": 0, "b": 0.0}
        ))
    store.add_mutation(MutationRecord(
        mutation_id=BRAF_GOF_ID,
        gene_id=673,
        gene_name="BRAF",
        pathways=["ERK", "OIS"],
        is_driver=True,
        effect="GOF",
    ))
    # DENSITY_MUT_ID : portée par tous les fondateurs et leurs descendants,
    # calibre le carrying capacity : N* = (b0 - d0) / (2 * weight * d)
    # Avec b0=0.8, d0=0.1, weight=2 → N* = 0.7 / (2 * 0.001) ≈ 350
    store.add_mutation(MutationRecord(
        mutation_id=DENSITY_MUT_ID, is_driver=False, extra={"d": 0.001, "b": 0.0}
    ))
    return store

store = build_store()
print(f"{len(store.all_ids())} mutations — {N_PASS} passengers, 1 BRAF GOF")

5002 mutations — 5000 passengers, 1 BRAF GOF


## 2. Engine

| Plugin | Target | Catégorie | Poids / alpha |
|---|---|---|---|
| `ErkPathwayPlugin` | `rate_function` | `proliferative` | w=2.0, α=0 |
| `MelanocyteOISPlugin` | `senescence` | — | w=1.0 |
| `DummyDeathPlugin` | `rate_function` | `deleterious` | w=1.0, α=1 (densité-dépendant) |
| `DummyBirthPlugin` | `rate_function` | `proliferative` | w=1.0, α=0 |
| `ConstantMutationRatePlugin` | `mutation_rate` | — | μ=1 |

In [13]:
N_FOUNDERS = 250  # 9 WT + 1 BRAF → le WT établit un plateau avant que BRAF prenne le dessus

def build_engine(store, seed=42, max_steps=8_000, max_cells=3_000):
    plugins = {
        "erk_pathway":    ErkPathwayPlugin(params={}, store=store),
        "melanocyte_ois": MelanocyteOISPlugin(params={}, store=store),
        "base_death":     DummyDeathPlugin(params={}, store=store),
        "base_birth":     DummyBirthPlugin(params={}, store=store),
        "mutation_rate":  ConstantMutationRatePlugin(params={"mu": 1.0}, store=store),
    }
    plugin_configs = {
        "erk_pathway": PluginConfig(
            name="erk_pathway", plugin_type="erk_pathway",
            target="rate_function", category="proliferative",
            weight=3.0, alpha=0,
        ),
        "melanocyte_ois": PluginConfig(
            name="melanocyte_ois", plugin_type="melanocyte_ois",
            target="senescence",
            weight=1.0, alpha=0,
        ),
        # "base_death": PluginConfig(
        #     name="base_death", plugin_type="base_death",
        #     target="rate_function", category="deleterious",
        #     weight=2.0, alpha=1,
        # ),
        "mutation_rate": PluginConfig(
            name="mutation_rate", plugin_type="mutation_rate",
            target="mutation_rate",
            weight=1.0, alpha=0,
        ),
    }
    rate_fn = WeightedSumRate(RateFunctionConfig(
        baseline_birth_rate=0.8,
        baseline_death_rate=0.1,
        carrying_capacity=500,

    ))
    return GillespieEngine(
        plugins=plugins,
        plugin_configs=plugin_configs,
        rate_function=rate_fn,
        mutation_generator=UniformMutationGenerator(store, params={"seed": seed}),
        store=store,
        recorder=Recorder(snapshot_interval=5, dump_final_state=True),
        config=SimulationConfig(
            max_steps=max_steps, max_time=1e9, max_cells=max_cells, seed=seed
        ),
    )


def make_founders(engine):
    founders = [engine.make_founder_cell() for _ in range(N_FOUNDERS)]
    for f in founders:
        f.state.mutations.add(DENSITY_MUT_ID)
    # founders[0] porte BRAF GOF ; les 9 autres sont WT purs
    founders[0].state.mutations.add(BRAF_GOF_ID)
    engine._recorder.record_driver(step=0, cell_id=founders[0].id, mutation_id=BRAF_GOF_ID)
    return founders

## 3. Run

In [14]:
engine = build_engine(store, seed=7, max_cells=5_000, max_steps=25_000)
result = engine.run(make_founders(engine))

snap = result.snapshots
print(f"Stop       : {result.stop_reason}")
print(f"Final N    : {snap[-1].n_alive if snap else 0}")
print(f"Divisions  : {len(result.divisions)}")
print(f"Deaths     : {len(result.deaths)}")
print(f"Sénescences: {len(result.senescence)}")
print(f"Drivers    : {len(result.drivers)}  (dont 1 fondateur BRAF GOF et 1 fondateur NRAS GOF à step 0)")
print()
# for ev in result.drivers:
#     rec = store.get(ev.mutation_id)
#     print(f"  step {ev.step:5d}  cell {ev.cell_id:4d}  {rec.gene_name} {rec.effect}")
exporter = SimulationExporter(result, store=engine._store, config=engine._config)
state = exporter._build_lineage()
ct = exporter._build_clone_tree(state)

for clone in ct['nodes']: 
    print(f"{clone['label']}\t{clone['final_size']}")

Stop       : zero_rate
Final N    : 1223
Divisions  : 4243
Deaths     : 3271
Sénescences: 1222
Drivers    : 1  (dont 1 fondateur BRAF GOF et 1 fondateur NRAS GOF à step 0)

Founding clone	1
BRAF GOF	1222


## 4. Export JSON

In [16]:
exporter = SimulationExporter(result, store=engine._store, config=engine._config)
exporter.save("erk_ois.evoseer_new_param_500_founders.json")
print("Exporté dans erk_ois.evoseer.json")

Exporté dans erk_ois.evoseer.json
